In [1]:
import reservoirpy as rpy
from reservoirpy.nodes import Reservoir, Ridge
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.makedirs("Classical_Reservoir_learning", exist_ok=True)

rpy.set_seed(0)

In [2]:
dff=pd.read_csv("dff.csv",header=0,index_col=0)
dff1= pd.read_csv("Data_raw.csv",header=0,index_col=0)
dff["STR"] = dff1["STR"];

In [3]:
dff=dff.fillna(0)

In [4]:
test = dff.iloc[-245:]

In [5]:
L=245 # 需要训练245个Reservoir
Total = 816 #总共有816个数据
#For entire sample forecasting (1997.08-2017.12)
WL=Total-L # rolling window length 训练每一个Reservoir的数据集大小 WL=windows_length
ws=0# rolling window start index ws=277 for subsample, ws=0 for entire sample

In [6]:
RV=np.array(dff['RV']).reshape(-1,1)

In [7]:
F = ["RV", "MKT", "diff_DP", "IP", "DEF","EP","SMB","diff_TB","HML","INF","STR"]
features = len(F)

In [8]:
Data=np.array(dff[F]).reshape(-1,features)

In [ ]:
def run_crl(feature_list, reservoir_sizes, K=3):
    """Run classical reservoir for each N, return dict of {N: (predictions, mse)}."""
    n_feat = len(feature_list)
    data_arr = np.array(dff[feature_list]).reshape(-1, n_feat)
    results = {}
    
    for N in reservoir_sizes:
        reservoir = Reservoir(N, input_dim=n_feat, lr=0.6, sr=0.9, input_scaling=0.1)  # Paper: 0.1
        predictions = np.zeros((L, 1))
        ws = 0
        
        # Warm-up call to initialize reservoir state (needed for reservoirpy 0.4+)
        _warmup = reservoir.run(np.zeros((K, n_feat)))
        
        while ws < L:
            train_states = np.empty((WL - K, reservoir.output_dim))
            for t in range(WL - K):
                X = np.array(data_arr[ws + t:ws + t + K, :]).reshape(-1, n_feat)
                reservoir.reset()
                train_states[t, :] = reservoir.run(X)[-1, :]
            
            train_y = RV[ws + K:ws + WL]
            readout = Ridge(ridge=1e-7)
            readout = readout.fit(train_states, train_y)
            
            X_test = np.array(data_arr[ws + WL - K:ws + WL]).reshape(-1, n_feat)
            reservoir.reset()
            Y_test_states = reservoir.run(X_test)[-1, :]
            pred = readout.run(Y_test_states.reshape(1, -1))
            predictions[ws, 0] = float(pred.flat[0])
            
            ws += 1
            if ws % 50 == 0:
                print(f"    Window {ws}/{L}")
        
        mse = np.mean((predictions - RV[WL:]) ** 2)
        results[N] = (predictions, mse)
        print(f"  N={N}, MSE={mse:.6f}")
    
    return results

In [ ]:
# --- Run 1: CRL (RV only, N=50 per paper) ---
print("Training CRL (base, RV-only, N=50) ...")
F_base = ["RV"]
results_crl = run_crl(F_base, [50])
best_N_crl = 50
predictions_crl = results_crl[best_N_crl][0]
np.savetxt(f"Classical_Reservoir_learning/best_CRL({best_N_crl})_predictions.csv",
           predictions_crl, delimiter=',')
print(f"Saved: Classical_Reservoir_learning/best_CRL({best_N_crl})_predictions.csv")

In [ ]:
# --- Run 2: CRLX (all 11 features, N=20 per paper) ---
print("Training CRLX (extended, 11 features, N=20) ...")
F_ext = ["RV", "MKT", "diff_DP", "IP", "DEF", "EP", "SMB", "diff_TB", "HML", "INF", "STR"]
results_crlx = run_crl(F_ext, [20])
best_N_crlx = 20
predictions_crlx = results_crlx[best_N_crlx][0]
np.savetxt(f"Classical_Reservoir_learning/best_CRLX({best_N_crlx})_predictions.csv",
           predictions_crlx, delimiter=',')
print(f"Saved: Classical_Reservoir_learning/best_CRLX({best_N_crlx})_predictions.csv")

In [ ]:
plt.figure(figsize=(10, 6))
plt.rcParams['font.size'] = 16
plt.title("CRL vs CRLX - Predicted vs Actual RV")
plt.xlabel("$Date$")
plt.plot(test.index, predictions_crl, label=f"CRL (RV, N={best_N_crl})", color="red")
plt.plot(test.index, predictions_crlx, label=f"CRLX (11feat, N={best_N_crlx})", color="green")
plt.plot(test.index, RV[WL:], label="Actual", color="blue")
plt.xticks(test.index[::24], rotation=-45)
plt.ylabel('Realized Volatility')
plt.tight_layout()
plt.legend()
plt.savefig("Classical_Reservoir_learning/crl_comparison.png")
plt.show()

In [13]:
def compute_qlike(forecasts, actuals):
    """
    Compute the QLIKE (Quasi-Likelihood) loss function for evaluating forecasting accuracy.
    forecasts: Forecasted variance (sigma squared from a model)
    actuals: Realized variance (actual observed variance)
    """
    # Using absolute values of forecasts and actuals
    forecasts = np.abs(forecasts)
    actuals = np.abs(actuals)

    # Calculate the ratio and ensure it's positive
    ratio = actuals / forecasts

    # Compute QLIKE
    qlike = np.sum(ratio - np.log(ratio) - 1)
    return qlike


def qlike_loss(y_true, y_hat):
  """
  This function computes the mean Quasi-Likelihood (QLIKE) loss function.

  Args:
      y_true (np.ndarray): True values of the variable.
      y_hat (np.ndarray): Predicted values of the variable.

  Returns:
      float: Mean QLIKE loss between the true and predicted values.
  """
  eps = np.finfo(float).eps  # Machine epsilon for numerical stability
  w = np.abs(y_true - y_hat) / (y_hat + eps)
  return np.mean(np.log(1 + w**2))

In [14]:
actual = RV[WL:]

print(f"--- CRL (base, RV-only, best N={best_N_crl}) ---")
mse1 = np.mean((predictions_crl - actual) ** 2)
print(f"  MSE: {mse1:.6f}")

print(f"\n--- CRLX (extended, 11 features, best N={best_N_crlx}) ---")
mse2 = np.mean((predictions_crlx - actual) ** 2)
print(f"  MSE: {mse2:.6f}")

--- CRL (base, RV-only, best N=60) ---
  MSE: 0.009415

--- CRLX (extended, 11 features, best N=30) ---
  MSE: 0.008788
